# 01 — Data Preprocessing
**Urban Safety Perception — DSC 148**

Generates a grid of San Diego locations, computes environmental features for each,
and creates a composite safety score as the prediction target.

### Inputs
- `data/raw/crime_2023.csv` — SDPD calls for service
- `data/raw/walkability.csv` — EPA National Walkability Index
- `data/raw/census_bg/` — Census TIGER block group boundaries
- `data/raw/streetlights.geojson` — SD streetlight locations

### Output
- `data/processed/modeling_df.csv` — grid of SD locations with features + safety label

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from pathlib import Path
from tqdm import tqdm

RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(exist_ok=True)
print('Libraries loaded!')

## 1. Generate San Diego Location Grid

In [ ]:
LAT_MIN, LAT_MAX = 32.55, 33.10
LON_MIN, LON_MAX = -117.28, -116.93
LAT_STEP = 0.0045
LON_STEP = 0.0055

lats = np.arange(LAT_MIN, LAT_MAX, LAT_STEP)
lons = np.arange(LON_MIN, LON_MAX, LON_STEP)
grid_points = [(lat, lon) for lat in lats for lon in lons]
grid_df = pd.DataFrame(grid_points, columns=['lat', 'lon'])
grid_df['location_id'] = [f'sd_{i}' for i in range(len(grid_df))]

grid_gdf = gpd.GeoDataFrame(
    grid_df,
    geometry=gpd.points_from_xy(grid_df['lon'], grid_df['lat']),
    crs='EPSG:4326'
)
print(f'Grid points: {len(grid_gdf)}')
grid_gdf.head(3)

## 2. Load & Process Crime Data

In [ ]:
crime = pd.read_csv(RAW / 'crime_2023.csv', low_memory=False)
print('Crime shape:', crime.shape)
crime_by_beat = crime.groupby('beat').size().reset_index(name='crime_count')
crime_by_beat['crime_density_norm'] = (
    crime_by_beat['crime_count'] - crime_by_beat['crime_count'].min()
) / (crime_by_beat['crime_count'].max() - crime_by_beat['crime_count'].min())
crime_by_beat['crime_score'] = 1 - crime_by_beat['crime_density_norm']
print('Beats:', len(crime_by_beat))
crime_by_beat.head()

## 3. Load Walkability + Census Block Groups

In [ ]:
walk = pd.read_csv(RAW / 'walkability.csv', low_memory=False)
walk_sd = walk[walk['GEOID10'].astype(str).str.startswith('06073')].copy()
walk_sd = walk_sd[['GEOID10', 'NatWalkInd']].copy()
walk_sd['GEOID10'] = walk_sd['GEOID10'].astype(str)
walk_sd['walk_score'] = (
    walk_sd['NatWalkInd'] - walk_sd['NatWalkInd'].min()
) / (walk_sd['NatWalkInd'].max() - walk_sd['NatWalkInd'].min())
print(f'SD block groups: {len(walk_sd)}')
walk_sd.head(3)

In [ ]:
shp_files = list((RAW / 'census_bg').glob('*.shp'))
print('Shapefiles found:', shp_files)
census = gpd.read_file(shp_files[0])
census_sd = census[census['COUNTYFP'] == '073'].copy()
census_sd = census_sd.to_crs('EPSG:4326')
census_sd['GEOID'] = census_sd['GEOID'].astype(str)
census_walk = census_sd.merge(walk_sd, left_on='GEOID', right_on='GEOID10', how='left')
print('Census+walk shape:', census_walk.shape)
print('Missing walk scores:', census_walk['walk_score'].isna().sum())

## 4. Load Streetlights

In [ ]:
lights = gpd.read_file(RAW / 'streetlights.geojson')
lights = lights.to_crs('EPSG:4326')
print('Streetlights shape:', lights.shape)
lights.head(3)

## 5. Spatial Join — Assign Walk Score to Grid Points

In [ ]:
grid_walk = gpd.sjoin(
    grid_gdf,
    census_walk[['geometry', 'walk_score', 'NatWalkInd']],
    how='left',
    predicate='within'
)
grid_walk = grid_walk.drop_duplicates(subset='location_id')
print('After walk join:', grid_walk.shape)
print('Missing walk scores:', grid_walk['walk_score'].isna().sum())
grid_walk[['location_id', 'lat', 'lon', 'walk_score']].head(3)

## 6. Compute Lighting Score (streetlights within 200m)

In [ ]:
grid_proj = grid_walk.to_crs('EPSG:32611')
lights_proj = lights.to_crs('EPSG:32611')
lights_geom = lights_proj.geometry

light_counts = []
for geom in tqdm(grid_proj.geometry, desc='Computing light density'):
    buf = geom.buffer(200)
    light_counts.append(lights_geom.within(buf).sum())

grid_walk = grid_walk.copy()
grid_walk['light_count'] = light_counts
grid_walk['light_score'] = (
    grid_walk['light_count'] - grid_walk['light_count'].min()
) / (grid_walk['light_count'].max() - grid_walk['light_count'].min())
print('Light score stats:')
print(grid_walk['light_score'].describe())

## 7. Assign Crime Score

In [ ]:
median_crime_score = crime_by_beat['crime_score'].median()
grid_walk['crime_score'] = median_crime_score
print(f'Assigned median crime score: {median_crime_score:.3f}')

## 8. Compute Composite Safety Score (Target Variable)

In [ ]:
grid_walk['walk_score'] = grid_walk['walk_score'].fillna(grid_walk['walk_score'].median())

grid_walk['safety_score'] = (
    0.50 * grid_walk['crime_score'] +
    0.25 * grid_walk['walk_score'] +
    0.25 * grid_walk['light_score']
)

threshold = grid_walk['safety_score'].median()
grid_walk['safe_label'] = (grid_walk['safety_score'] >= threshold).astype(int)

print('Safety score stats:')
print(grid_walk['safety_score'].describe())
print('\nClass balance:')
print(grid_walk['safe_label'].value_counts())

## 9. Save Modeling DataFrame

In [ ]:
modeling_df = grid_walk[[
    'location_id', 'lat', 'lon',
    'crime_score', 'walk_score', 'NatWalkInd',
    'light_count', 'light_score',
    'safety_score', 'safe_label'
]].copy()
modeling_df = modeling_df.dropna(subset=['walk_score'])
modeling_df.to_csv(PROCESSED / 'modeling_df.csv', index=False)
print(f'Saved! Shape: {modeling_df.shape}')
modeling_df.describe()

## 10. EDA Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
modeling_df['walk_score'].hist(ax=axes[0], bins=30, color='steelblue')
axes[0].set_title('Walkability Score Distribution')
modeling_df['light_score'].hist(ax=axes[1], bins=30, color='orange')
axes[1].set_title('Lighting Score Distribution')
modeling_df['safety_score'].hist(ax=axes[2], bins=30, color='green')
axes[2].set_title('Safety Score Distribution (Target)')
plt.tight_layout()
plt.savefig(PROCESSED / 'eda_distributions.png', dpi=150)
plt.show()
print('EDA plot saved!')

In [ ]:
corr_cols = ['crime_score', 'walk_score', 'light_score', 'safety_score']
corr = modeling_df[corr_cols].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(PROCESSED / 'eda_correlation.png', dpi=150)
plt.show()
print('Correlation matrix saved!')